# Chat Completion with Seed Control

This notebook demonstrates how to use the **seed** parameter and **temperature=0** with the OpenAI-compatible chat completion API to achieve **deterministic, reproducible** outputs.

## Why does this matter?

Large language models are inherently stochastic — given the same prompt, they can produce different responses each time. This is useful for creativity, but problematic when you need **consistent, repeatable results** (e.g., automated testing, benchmarking, or building reliable pipelines).

Two parameters work together to control this behavior:

- **`seed`** (integer): Initializes the random number generator used during token sampling. When you provide the same seed with the same prompt and parameters, the model attempts to produce the same output every time.
- **`temperature`** (float, 0.0–2.0): Controls the randomness of token selection. At **temperature=0**, the model always picks the most probable next token (greedy decoding), eliminating sampling randomness entirely.

**Together**, `seed=42` + `temperature=0` gives you the strongest reproducibility guarantee available. The seed alone is not sufficient if temperature is high, because the sampling process introduces randomness that the seed cannot fully control.

> **Dependencies** are managed by `uv` via `pyproject.toml`.
> Run `uv sync` once in this folder to create the `.venv`, then select
> the **training-notebooks** kernel in JupyterLab.

In [ ]:
import os
import json
import requests
from pathlib import Path

# Load .env from this directory (copy .env.sample to .env to configure)
_env_path = Path(".env")
if _env_path.exists():
    for line in _env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip())

# ---- Provider Configuration ----
# Set PROVIDER to "local" for Ollama or "openai" for the OpenAI API.
PROVIDER = os.environ.get("PROVIDER", "local")

# ---- Local Ollama settings ----
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
LOCAL_MODEL = os.environ.get("OLLAMA_MODEL", "phi3:mini")

# ---- OpenAI API settings ----
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
OPENAI_ENDPOINT = os.environ.get("OPENAI_ENDPOINT", "https://api.openai.com/v1/chat/completions")
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

print(f"Provider:  {PROVIDER}")
if PROVIDER == "local":
    print(f"Ollama URL: {OLLAMA_BASE_URL}")
    print(f"Model:      {LOCAL_MODEL}")
else:
    print(f"Endpoint:   {OPENAI_ENDPOINT}")
    print(f"Model:      {OPENAI_MODEL}")
    print(f"API Key:    {\'***\' + OPENAI_API_KEY[-4:] if len(OPENAI_API_KEY) > 4 else \'(not set)\'}")

In [ ]:
def chat_completion(prompt, seed=42, temperature=0):
    """
    Send a chat completion request with seed and temperature control.

    Args:
        prompt (str): The user message to send.
        seed (int): Seed for deterministic generation.
        temperature (float): Sampling temperature (0 = greedy/deterministic).

    Returns:
        dict: The full JSON response from the API.
    """
    messages = [{"role": "user", "content": prompt}]

    payload = {
        "messages": messages,
        "seed": seed,
        "temperature": temperature,
        "max_completion_tokens": 2048,
    }

    if PROVIDER == "local":
        url = f"{OLLAMA_BASE_URL}/v1/chat/completions"
        payload["model"] = LOCAL_MODEL
        headers = {"Content-Type": "application/json"}
    else:
        url = OPENAI_ENDPOINT
        payload["model"] = OPENAI_MODEL
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {OPENAI_API_KEY}",
        }

    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    return response.json()


def extract_content(response):
    """Extract the assistant message content from a chat completion response."""
    return response["choices"][0]["message"]["content"]


print("chat_completion() function defined successfully.")

In [ ]:
# ---- Experiment 1: Same seed + temperature=0 produces identical outputs ----

PROMPT = "List exactly 3 fun facts about the planet Mars. Be concise."

print("=" * 70)
print("EXPERIMENT 1: Reproducibility with seed=42, temperature=0")
print("=" * 70)
print(f"Prompt: \"{PROMPT}\"\n")

responses = []
for i in range(3):
    result = chat_completion(PROMPT, seed=42, temperature=0)
    content = extract_content(result)
    responses.append(content)
    print(f"--- Run {i + 1} ---")
    print(content)
    print()

# Check if all responses are identical
all_identical = all(r == responses[0] for r in responses)
print("=" * 70)
print(f"All 3 responses identical? {all_identical}")
print("=" * 70)

## What happened?

All three responses above should be **identical** (word-for-word). This is because:

1. We used the **same seed** (`42`) for every call, so the random number generator starts in the same state.
2. We set **temperature=0**, which removes all sampling randomness — the model always picks the single most likely token.

This combination gives us fully deterministic output: **same prompt + same seed + temperature=0 = same response every time**.

In [ ]:
# ---- Experiment 2: Different seeds produce different outputs ----

print("=" * 70)
print("EXPERIMENT 2: Different seeds produce different outputs")
print("=" * 70)
print(f"Prompt: \"{PROMPT}\"\n")

for seed_val in [42, 99]:
    result = chat_completion(PROMPT, seed=seed_val, temperature=0)
    content = extract_content(result)
    print(f"--- seed={seed_val} ---")
    print(content)
    print()

print("=" * 70)
print("Notice how changing the seed (42 vs 99) can alter the model's output,")
print("even though temperature is still 0.")
print("=" * 70)

## Seed changes the output

Even with `temperature=0`, changing the **seed** value alters which path the model takes through its probability space. The seed determines the initial state of the random number generator, which influences tie-breaking and minor sampling decisions within the model.

Key insight: the seed is like a "run ID" — each unique seed value gives you a **consistent but potentially different** output. Use the same seed to reproduce; use a different seed to get a new variation.

In [ ]:
# ---- Experiment 3: Higher temperature overrides determinism ----

print("=" * 70)
print("EXPERIMENT 3: temperature=0.8 introduces randomness (same seed=42)")
print("=" * 70)
print(f"Prompt: \"{PROMPT}\"\n")

high_temp_responses = []
for i in range(3):
    result = chat_completion(PROMPT, seed=42, temperature=0.8)
    content = extract_content(result)
    high_temp_responses.append(content)
    print(f"--- Run {i + 1} (seed=42, temperature=0.8) ---")
    print(content)
    print()

all_identical_high_temp = all(r == high_temp_responses[0] for r in high_temp_responses)
print("=" * 70)
print(f"All 3 responses identical? {all_identical_high_temp}")
if not all_identical_high_temp:
    print("As expected! Higher temperature introduces randomness that the seed")
    print("cannot fully constrain, leading to varied outputs.")
else:
    print("Interesting — some backends may still produce identical results with")
    print("a fixed seed even at higher temperatures. Results may vary by provider.")
print("=" * 70)

## Summary: Key Takeaways

| Setting | Behavior |
|---|---|
| `seed=42, temperature=0` | **Deterministic** — identical outputs every time |
| `seed=42` vs `seed=99`, `temperature=0` | **Different but individually reproducible** — each seed gives a consistent result |
| `seed=42, temperature=0.8` | **Non-deterministic** — temperature adds randomness that the seed cannot fully control |

### When to use seed control

- **Automated testing**: Pin a seed to get predictable outputs for assertions.
- **Benchmarking**: Compare model versions or prompts fairly by holding the seed constant.
- **Debugging**: Reproduce an exact output to investigate unexpected behavior.
- **Production pipelines**: Use `seed + temperature=0` when consistency matters more than variety.

### Caveats

- Determinism is **best-effort** on some providers. OpenAI notes that identical outputs are not 100% guaranteed across all infrastructure changes.
- Ollama with local models generally provides strong determinism with `seed + temperature=0`.
- Always combine `seed` with `temperature=0` for the strongest reproducibility guarantee.